# Starting weights, part 2: ResNet50 arm + replication on the EyePACS confirmation set

Follows `04_results/starting_weights_part2_plan.json` (written before these models existed).

1. **Train ResNet50 from two new starting points**, canonical CNN recipe (15 epochs, AdamW lr 1e-4, wd 1e-4, cosine, batch 32), 3 seeds each: semi-weakly supervised `fb_swsl_ig1b_ft_in1k` and self-supervised DINO ResNet-50. The existing `resnet50_5class` (supervised `a1_in1k`) is the third arm.
2. **Run the 12 new models** (6 ViT-Base from the first experiment, 6 ResNet50) on the EyePACS confirmation set.
3. **Analyse on the confirmation set** (15,794 eyes, 25% referral, patient-cluster bootstrap): seed siblings vs same architecture with different starting weights vs different architectures, for referable disease, grade 3+, PDR; strength-adjusted; matched-volume benefit.

A100, High-RAM. About 60 to 75 minutes (training ~30 min, image extraction and inference ~20 min, analysis ~10 min). Training and inference are resumable.

In [1]:
# Cell 1: setup
from google.colab import auth, drive
auth.authenticate_user(); drive.mount('/content/drive')
!pip -q install timm
import io, os, json, time, zipfile, glob, datetime, random, math, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
svc = gbuild('drive', 'v3')
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, CKPT, PREDS, RESULTS, AUDIT = (f'{BASE}/{d}' for d in ('01_splits','02_checkpoints','03_predictions','04_results','99_audit'))
SPLIT_IDS = {'train': '1FU30c6vRYMLqX8h9z2F58iEqow1Zx8NQ', 'val': '1Z9tFlAGe0j4RVRgG9-6LnhTrukvFy87y', 'test': '1mSlAqeNtFJ0Ap6BMenrrQxHaWep-9nEj'}
IMAGE_ZIPS = {'batch1': '11WspNNHS73OadbW-sqy7BSzcrsBkX1d6', 'batch2': '1zG7rHjm8Ty9LHG4LqMzgnOzrT_90VO9H', 'batch3': '1L88vUNSxWjdGPBquhknZhafcnXXRd3ES',
              'batch4': '1-FrpGB74qEGzDvXlA9FhmB2OeeZkKPAN', 'batch5': '1mEIrSGVExZ6zmxnRLxyFab5yB1kH5YBT'}
DINO_RN50_URL = 'https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth'
NUM_WORKERS = min(8, os.cpu_count())
def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid), chunksize=50 * 1024 * 1024); done = False
        while not done: _, done = d.next_chunk()
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# Cell 2: ResNet50 starting points, with preflight
DINO_SD = torch.hub.load_state_dict_from_url(DINO_RN50_URL, map_location='cpu')
def build_resnet(init, nc=5, pretrained=True):
    if init == 'swsl':
        return timm.create_model('resnet50.fb_swsl_ig1b_ft_in1k', pretrained=pretrained, num_classes=nc)
    if init == 'dino':
        m = timm.create_model('resnet50', pretrained=False, num_classes=nc)
        if pretrained:
            missing, unexpected = m.load_state_dict(DINO_SD, strict=False)
            assert set(missing) <= {'fc.weight', 'fc.bias'} and not [u for u in unexpected if not u.startswith('fc.')], (missing, unexpected)
        return m
    raise ValueError(init)
for init in ('swsl', 'dino'):
    m = build_resnet(init); n = sum(p.numel() for p in m.parameters())
    with torch.no_grad(): o = m(torch.zeros(2, 3, 224, 224))
    assert o.shape == (2, 5) and 2.3e7 < n < 2.6e7
    print(f'preflight resnet50 {init}: {n/1e6:.1f}M params, output {tuple(o.shape)}')
RN_MODELS = [{'name': f'resnet50_{k}_5class', 'arch': 'resnet50', 'init': k, 'nc': 5, 'epochs': 15, 'bs': 32, 'lr': 1e-4, 'wd': 1e-4} for k in ('swsl', 'dino')]
REPLICATES = [0, 1, 2]; BASE_SEED = 20261301; IMG_SIZE = 224
def tag(k): return '' if k == 0 else f'_seed{k}'

Downloading: "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_resnet50_pretrain.pth


100%|██████████| 90.0M/90.0M [00:02<00:00, 41.9MB/s]
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

preflight resnet50 swsl: 23.5M params, output (2, 5)
preflight resnet50 dino: 23.5M params, output (2, 5)


In [3]:
# Cell 3: EyePACS training images and splits (same source as the canonical notebooks)
os.makedirs('/content/img', exist_ok=True); splits = {}
for s, fid in SPLIT_IDS.items():
    p = f'split_{s}.csv'
    if not os.path.exists(p): dl(fid, p)
    splits[s] = pd.read_csv(p)
if not glob.glob('/content/img/**/*.jp*g', recursive=True):
    for name, fid in IMAGE_ZIPS.items():
        z = f'/content/{name}.zip'; print('downloading', name); dl(fid, z)
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/img')
        os.remove(z)
paths = glob.glob('/content/img/**/*.jp*g', recursive=True) + glob.glob('/content/img/**/*.png', recursive=True)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}
assert not (set().union(*[set(d.image_id.astype(str)) for d in splits.values()]) - set(INDEX)), 'EyePACS training images missing'
print({s: len(d) for s, d in splits.items()})

downloading batch1
downloading batch2
downloading batch3
downloading batch4
downloading batch5
{'train': 13532, 'val': 2900, 'test': 2900}


In [4]:
# Cell 4: canonical CNN training loop
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                               transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
class DR(Dataset):
    def __init__(self, df, tf, index): self.df = df.reset_index(drop=True); self.tf = tf; self.index = index
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(self.index[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i
def seed_everything(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def predict(model, df, dev, index, bs=128):
    model.eval(); probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in DataLoader(DR(df, TF_EVAL, index), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True):
        out = model(x.to(dev, non_blocking=True)).float(); probs[idx.numpy()] = torch.softmax(out, 1)[:, 2:].sum(1).double().cpu().numpy()
    return probs
def train_one(cfg, dev, seed):
    seed_everything(seed); model = build_resnet(cfg['init']).to(dev)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(splits['train'], TF_TRAIN, INDEX), batch_size=cfg['bs'], shuffle=True, num_workers=NUM_WORKERS, drop_last=True, generator=g,
                     pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd']); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    lossf = nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler('cuda'); best = (-1.0, None, -1)
    for ep in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True); y = lvl.long().to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += float(loss.item())
        sch.step(); vauc = float(roc_auc_score(splits['val'].binary_label, predict(model, splits['val'], dev, INDEX)))
        star = ''
        if vauc > best[0]: best = (vauc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep + 1); star = '  <- best'
        print(f"  epoch {ep+1}/{cfg['epochs']}  loss {tot/len(dl_):.4f}  val_auc {vauc:.4f}  ({time.time()-t0:.0f}s){star}")
    model.load_state_dict(best[1]); return model, best[0], best[2]
print('ready')

ready


In [5]:
# Cell 5: train 6 ResNet50 models (resumable)
dev = 'cuda'; RECORD = f'{AUDIT}/starting_weights_part2_record.json'
rec = json.load(open(RECORD)) if os.path.exists(RECORD) else {'started': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'models': {}}
todo = [(mi, c, k) for k in REPLICATES for mi, c in enumerate(RN_MODELS) if not os.path.exists(f"{CKPT}/eyepacs_{c['name']}{tag(k)}_v2.pt")]
print('to train:', [c['name'] + tag(k) for _, c, k in todo])
for i, (mi, cfg, k) in enumerate(todo, 1):
    seed = BASE_SEED + 100 * mi + k; name = cfg['name'] + tag(k); t0 = time.time(); print(f'\n=== [{i}/{len(todo)}] {name} (seed {seed}) ===')
    model, vauc, ep = train_one(cfg, dev, seed); ck = f'{CKPT}/eyepacs_{name}_v2.pt'
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed, 'replicate': k, 'best_val_auc': vauc, 'best_epoch': ep}, ck)
    tauc = float(roc_auc_score(splits['test'].binary_label, predict(model, splits['test'], dev, INDEX)))
    rec['models'][name] = {'seed': seed, 'best_val_auc': round(vauc, 4), 'best_epoch': ep, 'test_auc': round(tauc, 4), 'minutes': round((time.time()-t0)/60, 1), 'checkpoint_sha256': sha256_file(ck)}
    json.dump(rec, open(RECORD, 'w'), indent=2); print(f'  done: val {vauc:.4f} (epoch {ep}) | test {tauc:.4f}'); del model; torch.cuda.empty_cache()

to train: ['resnet50_swsl_5class', 'resnet50_dino_5class', 'resnet50_swsl_5class_seed1', 'resnet50_dino_5class_seed1', 'resnet50_swsl_5class_seed2', 'resnet50_dino_5class_seed2']

=== [1/6] resnet50_swsl_5class (seed 20261301) ===
  epoch 1/15  loss 0.7501  val_auc 0.8409  (99s)  <- best
  epoch 2/15  loss 0.6500  val_auc 0.8543  (37s)  <- best
  epoch 3/15  loss 0.6163  val_auc 0.8736  (38s)  <- best
  epoch 4/15  loss 0.5911  val_auc 0.8654  (38s)
  epoch 5/15  loss 0.5709  val_auc 0.8805  (38s)  <- best
  epoch 6/15  loss 0.5515  val_auc 0.8947  (37s)  <- best
  epoch 7/15  loss 0.5274  val_auc 0.8993  (38s)  <- best
  epoch 8/15  loss 0.5018  val_auc 0.8915  (38s)
  epoch 9/15  loss 0.4758  val_auc 0.8915  (37s)
  epoch 10/15  loss 0.4455  val_auc 0.8894  (38s)
  epoch 11/15  loss 0.4102  val_auc 0.8963  (38s)
  epoch 12/15  loss 0.3700  val_auc 0.8931  (38s)
  epoch 13/15  loss 0.3486  val_auc 0.8960  (38s)
  epoch 14/15  loss 0.3141  val_auc 0.8931  (38s)
  epoch 15/15  loss 0.30

In [6]:
# Cell 6: confirmation-set images, then inference for the 12 new models (skips any already done)
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str})
gcol = next(c for c in ('level','grade') if c in fr.columns)
FR = pd.DataFrame({'image_id': fr.image_id.astype(str), 'level': fr[gcol].astype(int), 'patient_id': fr.patient_id.astype(str)})
FR['binary_label'] = (FR.level >= 2).astype(int)
os.makedirs('/content/fresh', exist_ok=True)
if len(glob.glob('/content/fresh/**/*.*', recursive=True)) < len(FR):
    for z in sorted(glob.glob(f'{BASE}/06_fresh_images/eyepacs_fresh_batch*_v2.zip')):
        print('extracting', os.path.basename(z))
        with zipfile.ZipFile(z) as zf: zf.extractall('/content/fresh')
FIDX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/fresh/**/*.*', recursive=True) if p.lower().endswith(('.png','.jpg','.jpeg'))}
assert not (set(FR.image_id) - set(FIDX)), f'{len(set(FR.image_id) - set(FIDX))} confirmation-set images missing'
NEW = [('vit_base_mae_5class', 'vit'), ('vit_base_dinov2_5class', 'vit'), ('resnet50_swsl_5class', 'rn'), ('resnet50_dino_5class', 'rn')]
for name, kind in NEW:
    for k in REPLICATES:
        out = f'{PREDS}/eyepacs_predictions_fresh_{name}{tag(k)}_v2.csv'
        if os.path.exists(out): continue
        blob = torch.load(f'{CKPT}/eyepacs_{name}{tag(k)}_v2.pt', map_location='cpu', weights_only=False); cfg = blob['config']
        m = timm.create_model(cfg['arch'], pretrained=False, num_classes=5, img_size=224) if kind == 'vit' else timm.create_model('resnet50', pretrained=False, num_classes=5)
        m.load_state_dict(blob['state_dict'], strict=True); m = m.to('cuda')
        pr = predict(m, FR, 'cuda', FIDX)
        pd.DataFrame({'image_id': FR.image_id, 'true_grade': FR.level, 'predicted_probability': pr, 'model_name': name, 'replicate': k}).to_csv(out, index=False, float_format='%.17g')
        print('fresh-set predictions:', name + tag(k), f'AUC {roc_auc_score(FR.binary_label, pr):.4f}'); del m; torch.cuda.empty_cache()

extracting eyepacs_fresh_batch1_v2.zip
extracting eyepacs_fresh_batch2_v2.zip
extracting eyepacs_fresh_batch3_v2.zip
extracting eyepacs_fresh_batch4_v2.zip
fresh-set predictions: vit_base_mae_5class AUC 0.8678
fresh-set predictions: vit_base_mae_5class_seed1 AUC 0.8651
fresh-set predictions: vit_base_mae_5class_seed2 AUC 0.8671
fresh-set predictions: vit_base_dinov2_5class AUC 0.9193
fresh-set predictions: vit_base_dinov2_5class_seed1 AUC 0.9113
fresh-set predictions: vit_base_dinov2_5class_seed2 AUC 0.9078
fresh-set predictions: resnet50_swsl_5class AUC 0.9061
fresh-set predictions: resnet50_swsl_5class_seed1 AUC 0.9124
fresh-set predictions: resnet50_swsl_5class_seed2 AUC 0.9023
fresh-set predictions: resnet50_dino_5class AUC 0.8924
fresh-set predictions: resnet50_dino_5class_seed1 AUC 0.8969
fresh-set predictions: resnet50_dino_5class_seed2 AUC 0.8911


In [7]:
# Cell 7: analysis on the confirmation set
from scipy.stats import rankdata
VIT = ['vit_base_5class', 'vit_base_mae_5class', 'vit_base_dinov2_5class']; RN = ['resnet50_5class', 'resnet50_swsl_5class', 'resnet50_dino_5class']
OTHER = ['densenet121_5class', 'efficientnet_b3_5class', 'retfound_5class']; VARIANTS = VIT + RN + OTHER; SEEDS = [0, 1, 2]
ORIG = {'vit_base_5class', 'resnet50_5class', 'densenet121_5class', 'efficientnet_b3_5class', 'retfound_5class'}
FAMILY = {**{v: 'vit' for v in VIT}, **{v: 'resnet' for v in RN}, 'densenet121_5class': 'densenet', 'efficientnet_b3_5class': 'effnet', 'retfound_5class': 'retfound'}
LABEL = {'vit_base_5class': 'ViT sup', 'vit_base_mae_5class': 'ViT MAE', 'vit_base_dinov2_5class': 'ViT DINOv2', 'resnet50_5class': 'ResNet sup',
         'resnet50_swsl_5class': 'ResNet SWSL', 'resnet50_dino_5class': 'ResNet DINO', 'densenet121_5class': 'DenseNet', 'efficientnet_b3_5class': 'EffNet', 'retfound_5class': 'RETFound'}
FRs = FR.sort_values('image_id').reset_index(drop=True); IDS = FRs.image_id.to_numpy(); G = FRs.level.to_numpy(); n = len(IDS)
codes, cidx = np.unique(FRs.patient_id, return_inverse=True); ncl = len(codes)
P = {}
for v in VARIANTS:
    for k in SEEDS:
        df = pd.read_csv(f'{PREDS}/eyepacs_predictions_fresh_{v}{tag(k)}_v2.csv', dtype={'image_id': str})
        pc = next(c for c in ('predicted_probability','probability','prob') if c in df.columns)
        s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(IDS); assert s.notna().all(), (v, k); P[(v, k)] = s.to_numpy()
refer_ = lambda s, f: (lambda m: (m.__setitem__(np.argsort(-s, kind='stable')[:int(round(f * len(s)))], True), m)[1])(np.zeros(len(s), bool))
F = 0.25; REF = {key: refer_(v, F) for key, v in P.items()}; RK = {key: rankdata(v) / n for key, v in P.items()}
referable = G >= 2; TARGETS = {'referable': G >= 2, 'grade3plus': G >= 3, 'pdr': G == 4}
QUAL = {v: {'auc': round(float(np.mean([roc_auc_score(referable, P[(v, k)]) for k in SEEDS])), 4),
            'referable_missed': round(float(np.mean([(referable & ~REF[(v, k)]).sum() for k in SEEDS])), 1)} for v in VARIANTS}
print(pd.DataFrame(QUAL).T.to_string())
def combos(a, b): return [((a, i), (b, j)) for i in SEEDS for j in SEEDS if (a != b or i < j)]
def ptype(a, b):
    if a == b: return 'seed_siblings'
    if FAMILY[a] == FAMILY[b]: return 'same_arch_diff_init_' + FAMILY[a]
    if a in ORIG and b in ORIG: return 'different_architecture'
    return 'new_init_vs_other_architecture'
UNORD = [(a, b) for i, a in enumerate(VARIANTS) for b in VARIANTS[i:]]
TYPES = ['seed_siblings', 'same_arch_diff_init_vit', 'same_arch_diff_init_resnet', 'different_architecture', 'new_init_vs_other_architecture']
rng = np.random.default_rng(20260929); BW = [np.bincount(rng.integers(0, ncl, ncl), minlength=ncl)[cidx].astype(float) for _ in range(1000)]
RES, PAIRVAL = {}, {}
for tname, tgt in TARGETS.items():
    VEC = {}
    for a, b in UNORD:
        cs = combos(a, b); ma = mb = both = 0
        for ka, kb in cs:
            x, y = tgt & ~REF[ka], tgt & ~REF[kb]; ma = ma + x; mb = mb + y; both = both + (x & y)
        VEC[(a, b)] = (ma / len(cs), mb / len(cs), both / len(cs))
    def ratio(p, w):
        ma, mb, both = (x @ w for x in VEC[p]); T = tgt @ w
        return both / (ma * mb / T) if ma * mb > 0 else np.nan
    def summ(w):
        r = {t: float(np.nanmean([ratio(p, w) for p in UNORD if ptype(*p) == t])) for t in TYPES}
        r['same_arch_diff_init_pooled'] = float(np.nanmean([ratio(p, w) for p in UNORD if ptype(*p).startswith('same_arch')])); return r
    obs = summ(np.ones(n)); bs = [summ(w) for w in BW]
    ci = lambda key: [round(float(np.nanpercentile([b[key] for b in bs], 2.5)), 2), round(float(np.nanpercentile([b[key] for b in bs], 97.5)), 2)]
    out = {k: {'mean': round(v, 2), 'ci95': ci(k)} for k, v in obs.items()}
    for a_, b_ in (('seed_siblings','same_arch_diff_init_vit'), ('seed_siblings','same_arch_diff_init_resnet'), ('seed_siblings','same_arch_diff_init_pooled'),
                   ('same_arch_diff_init_vit','different_architecture'), ('same_arch_diff_init_resnet','different_architecture'), ('same_arch_diff_init_pooled','different_architecture')):
        d = [b[a_] - b[b_] for b in bs]
        out[f'{a_}__minus__{b_}'] = {'mean': round(obs[a_] - obs[b_], 2), 'ci95': [round(float(np.nanpercentile(d, 2.5)), 2), round(float(np.nanpercentile(d, 97.5)), 2)]}
    RES[tname] = out; PAIRVAL[tname] = {f'{LABEL[a]} | {LABEL[b]}': round(ratio((a, b), np.ones(n)), 2) for a, b in UNORD}
    print(f'\n=== co-miss ratio, {tname} ==='); print(pd.DataFrame(out).T.to_string())

                           auc  referable_missed
vit_base_5class         0.8619             894.7
vit_base_mae_5class     0.8667             904.0
vit_base_dinov2_5class  0.9128             616.7
resnet50_5class         0.8612             925.3
resnet50_swsl_5class    0.9070             661.3
resnet50_dino_5class    0.8934             755.3
densenet121_5class      0.8856             786.7
efficientnet_b3_5class  0.8604             920.0
retfound_5class         0.8765             824.0

=== co-miss ratio, referable ===
                                                           mean          ci95
seed_siblings                                              3.05  [2.91, 3.21]
same_arch_diff_init_vit                                    2.54  [2.43, 2.67]
same_arch_diff_init_resnet                                 2.64  [2.52, 2.78]
different_architecture                                     2.38  [2.28, 2.49]
new_init_vs_other_architecture                             2.61  [2.49, 2.73]
same_arc

In [8]:
# Cell 8: strength adjustment, benefit, write
rows = [{'type': ptype(a, b), 'log_c': np.log(PAIRVAL['referable'][f'{LABEL[a]} | {LABEL[b]}']),
         'mean_m': (QUAL[a]['referable_missed'] + QUAL[b]['referable_missed']) / 2, 'gap_m': abs(QUAL[a]['referable_missed'] - QUAL[b]['referable_missed'])}
        for a, b in UNORD if a != b]
PR = pd.DataFrame(rows); PR['type'] = PR.type.where(~PR.type.str.startswith('same_arch'), 'same_arch_diff_init')
X = pd.get_dummies(PR.type).astype(float); X['mean_m'] = PR.mean_m; X['gap_m'] = PR.gap_m
beta, *_ = np.linalg.lstsq(X.values, PR.log_c.values, rcond=None); B = dict(zip(X.columns, beta))
ADJ = {'same_arch_diff_init_vs_different_architecture_ratio': round(float(np.exp(B['same_arch_diff_init'] - B['different_architecture'])), 3),
       'n_pairs': int(len(PR)), 'note': 'multiplicative difference in referable co-miss ratio, pair-level OLS on log ratio with pair mean and gap in referable misses; descriptive'}
ben = []
for a in VARIANTS:
    for b in VARIANTS:
        if a == b: continue
        mv = [(referable & ~REF[ka]).sum() - (referable & ~refer_(np.maximum(RK[ka], RK[kb]), F)).sum() for ka, kb in combos(a, b)]
        ben.append({'type': ptype(a, b), 'anchor': LABEL[a], 'partner': LABEL[b], 'mv_net': float(np.mean(mv))})
BEN = pd.DataFrame(ben); BYT = BEN.groupby('type').mv_net.mean().round(2).to_dict()
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'starting_weights_part2_plan.json',
       'status': 'exploratory; confirmation set previously opened under lock v3', 'referral_fraction': F, 'quality': QUAL,
       'co_miss_ratio': RES, 'pair_co_miss_referable': PAIRVAL['referable'], 'strength_adjusted': ADJ, 'matched_volume_net_referable_by_type': BYT,
       'signals': {'vit_init_matters': RES['referable']['seed_siblings__minus__same_arch_diff_init_vit']['ci95'][0] > 0,
                   'resnet_init_matters': RES['referable']['seed_siblings__minus__same_arch_diff_init_resnet']['ci95'][0] > 0,
                   'architecture_matters_more': RES['referable']['same_arch_diff_init_pooled__minus__different_architecture']['ci95'][0] > 0}}
with open(f'{RESULTS}/starting_weights_part2_results.json', 'w') as f: json.dump(res, f, indent=2, default=float)
print('strength-adjusted:', ADJ); print('matched-volume net referable by type:', BYT); print('signals:', res['signals'])
print('Wrote 04_results/starting_weights_part2_results.json')

strength-adjusted: {'same_arch_diff_init_vs_different_architecture_ratio': 1.036, 'n_pairs': 36, 'note': 'multiplicative difference in referable co-miss ratio, pair-level OLS on log ratio with pair mean and gap in referable misses; descriptive'}
matched-volume net referable by type: {'different_architecture': 75.69, 'new_init_vs_other_architecture': 79.23, 'same_arch_diff_init_resnet': 77.37, 'same_arch_diff_init_vit': 88.0}
signals: {'vit_init_matters': True, 'resnet_init_matters': True, 'architecture_matters_more': True}
Wrote 04_results/starting_weights_part2_results.json
